In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/my_volume/BigMart Sales.csv")
)

In [0]:

df = (df
        .dropDuplicates()
        .fillna({"Item_Weight":0, "Outlet_Size": "Unknown"})
        .withColumn("Item_Fat_Content",
                     when(col("Item_Fat_Content").isin("LF", "low fat"),"Low Fat")
                    .when(col("Item_Fat_Content").isin("reg"), "Regular")
                    .otherwise(col("Item_Fat_Content"))
                    )
        
    )

print("Cleaned Rows are:", df.count())
df.display()

**  3. For each Outlet_Identifier, calculates:
Total Sales,
Average Sales,
Number of Unique Items Sold,
Total Sales from High MRP items only (Item_MRP >= 150) 
**

In [0]:
outlet_Summary = (
    df.groupBy("Outlet_Identifier")
    .agg(
        round(sum("Item_Outlet_Sales"), 2).alias("Total Sales"),
        round(avg("Item_Outlet_Sales"), 2).alias("Average Sales"),
        countDistinct("Item_Identifier").alias("Unique Items Sold"),
        round(sum(when(col("Item_MRP")>=150,col("Item_Outlet_Sales")).otherwise(0)), 2).alias("High MRP Sales")

    )
).display()